In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, length, when, count, avg, round as spark_round

# spark i baslatiyorum
spark = SparkSession.builder \
    .appName('TwitterAnalysis') \
    .getOrCreate()

print("Spark basladi!")

dosya = '../data/processed/cleaned_tweets.csv'

# csv yi pyspark dataframe olarak oku
# quote, escape ve multiLine parametreleri tweet metinlerindeki virgul ve satirbasi sorunlarini onluyor
spark_df = spark.read.csv(
    dosya,
    header=True,
    inferSchema=True,
    quote='"',
    escape='"',
    multiLine=True
)

# semasina bakalim hangi sutun ne tipte
spark_df.printSchema()

In [ ]:
print("--- 1. Donusum (Transformation) Islemleri ---")

# islem 1: karmasik filter + select
# United veya American havayoluna ait, negatif, guven skoru 0.9 ustu tweetler
print("\nIslem 1: United/American negatif + guven>0.9 (complex filter & select):")
filtreli = spark_df.filter(
    (col("airline").isin("United", "American")) &
    (col("airline_sentiment") == "negative") &
    (col("airline_sentiment_confidence") > 0.9)
).select("airline", "negativereason", "airline_sentiment_confidence", "text")
filtreli.show(3, truncate=50)

# islem 2: withColumn + when/otherwise - guven skoruna gore kategori sutunu olustur
# 1.0 ise "kesin", 0.8-1.0 arasi "yuksek", altiysa "dusuk" diye etiketle
print("\nIslem 2: Guven kategorisi sutunu (withColumn + when/otherwise):")
df_kategorili = spark_df.withColumn(
    "guven_kategori",
    when(col("airline_sentiment_confidence") == 1.0, "kesin")
    .when(col("airline_sentiment_confidence") >= 0.8, "yuksek")
    .otherwise("dusuk")
)
df_kategorili.select("airline", "airline_sentiment_confidence", "guven_kategori").show(5)

# islem 3: groupBy + birden fazla aggregation fonksiyonu ayni anda
# havayoluna gore tweet sayisi, ortalama guven, max retweet
print("\nIslem 3: Havayolu bazli coklu istatistik (groupBy + multiple agg):")
coklu_agg = spark_df.groupBy("airline").agg(
    count("*").alias("tweet_sayisi"),
    spark_round(avg("airline_sentiment_confidence"), 3).alias("ort_guven"),
    avg("retweet_count").alias("ort_rt")
).orderBy(col("tweet_sayisi").desc())
coklu_agg.show()

In [ ]:
print("--- 2. Eylem (Action) Islemleri ---")

# islem 4: count + distinct count birlikte
toplam = spark_df.count()
benzersiz_kullanici = spark_df.select("name").distinct().count()
benzersiz_konum = spark_df.filter(col("tweet_location") != "Bilinmiyor").select("tweet_location").distinct().count()
print(f"Toplam kayit: {toplam}")
print(f"Benzersiz kullanici: {benzersiz_kullanici}")
print(f"Benzersiz konum (bilinmiyor haric): {benzersiz_konum}\n")

# islem 5: describe - birden fazla numerik sutun icin istatistik
print("Birden fazla sutun icin istatistik (describe):")
spark_df.describe(['airline_sentiment_confidence', 'retweet_count']).show()

In [ ]:
print("--- 3. Spark SQL Islemleri ---")

# gecici tablo olustur
spark_df.createOrReplaceTempView('tweets_table')

# islem 6: CASE WHEN ile duygu etiketine gore turkce aciklama sutunu olustur
# + GROUP BY + HAVING (sadece 1000 ustu tweet olan havayollarini goster)
print("\nSQL Sorgu 1: CASE WHEN + HAVING - buyuk havayollarinin duygu dagilimi:")
sql1 = spark.sql("""
    SELECT
        airline,
        SUM(CASE WHEN airline_sentiment = 'negative' THEN 1 ELSE 0 END) as negatif,
        SUM(CASE WHEN airline_sentiment = 'positive' THEN 1 ELSE 0 END) as pozitif,
        SUM(CASE WHEN airline_sentiment = 'neutral' THEN 1 ELSE 0 END) as notr,
        COUNT(*) as toplam
    FROM tweets_table
    GROUP BY airline
    HAVING COUNT(*) > 500
    ORDER BY negatif DESC
""")
sql1.show()

# islem 7: ic ice sorgu (subquery) - her havayolunun negatif tweet oranini hesapla
# ve genel ortalamadan kotu olanlari bul
print("\nSQL Sorgu 2: Negatif tweet orani genel ortalamanin ustunde olan havayollari (subquery):")
sql2 = spark.sql("""
    SELECT
        airline,
        ROUND(negatif_oran, 2) as negatif_yuzde
    FROM (
        SELECT
            airline,
            (SUM(CASE WHEN airline_sentiment = 'negative' THEN 1 ELSE 0 END) * 100.0 / COUNT(*)) as negatif_oran
        FROM tweets_table
        GROUP BY airline
    ) alt_sorgu
    WHERE negatif_oran > (
        SELECT SUM(CASE WHEN airline_sentiment = 'negative' THEN 1 ELSE 0 END) * 100.0 / COUNT(*)
        FROM tweets_table
    )
    ORDER BY negatif_yuzde DESC
""")
sql2.show()

In [ ]:
import pandas as pd
import time

print("--- 4. Pandas vs PySpark Performans Karsilastirmasi ---")

# ayni veriyi pandas ile de okuyalim
pandas_df = pd.read_csv('/content/drive/MyDrive/Buyuk_Veri_Donem_Projesi/twitter_big_data_pipeline/data/processed/cleaned_tweets.csv')

# pandas testi - havayoluna gore negatif tweet sayma + ortalama guven
t1 = time.time()
pandas_sonuc = pandas_df[pandas_df['airline_sentiment'] == 'negative'].groupby('airline').agg(
    adet=('airline_sentiment', 'count'),
    ort_guven=('airline_sentiment_confidence', 'mean')
)
t2 = time.time()
pandas_sure = t2 - t1

# pyspark testi - ayni islem
t3 = time.time()
spark_sonuc = spark_df.filter(col("airline_sentiment") == "negative") \
    .groupBy("airline") \
    .agg(
        count("*").alias("adet"),
        avg("airline_sentiment_confidence").alias("ort_guven")
    ).collect()
t4 = time.time()
spark_sure = t4 - t3

print(f"Pandas suresi:  {pandas_sure:.5f} sn")
print(f"PySpark suresi: {spark_sure:.5f} sn")

print("\nNot: ~14.000 satirlik kucuk bir veride Pandas daha hizli cikabilir.")
print("PySpark'in asil gucu milyonlarca satirlik verilerde dagitik islemede ortaya cikar.")

spark.stop()